In [25]:
import spacy 
from spacy.training.example import Example
from spacy.util import minibatch
import random

In [26]:
train_data = [
    ("What is the price of rice?", {"entities": [(21, 25, "PRODUCT")]}),
    ("How much does sugar cost?", {"entities": [(14, 19, "PRODUCT")]}),
    ("Tell me the price of milk", {"entities": [(20, 24, "PRODUCT")]}),
    ("What is the cost of bread?", {"entities": [(20, 25, "PRODUCT")]}),
    ("How much is cooking oil?", {"entities": [(12, 23, "PRODUCT")]}),

    ("What is the price of maize flour?", {"entities": [(21, 32, "PRODUCT")]}),
    ("How much does salt cost?", {"entities": [(14, 18, "PRODUCT")]}),
    ("Tell me the cost of tea leaves", {"entities": [(20, 30, "PRODUCT")]}),
    ("What is the price of coffee?", {"entities": [(21, 27, "PRODUCT")]}),
    ("How much is bottled water?", {"entities": [(12, 25, "PRODUCT")]}),

    ("What is the price of a laptop?", {"entities": [(23, 29, "PRODUCT")]}),
    ("How much does a phone cost?", {"entities": [(16, 21, "PRODUCT")]}),
    ("Tell me the price of headphones", {"entities": [(20, 30, "PRODUCT")]}),
    ("What is the cost of a keyboard?", {"entities": [(20, 28, "PRODUCT")]}),
    ("How much is a mouse?", {"entities": [(12, 17, "PRODUCT")]}),

    ("What is the price of shoes?", {"entities": [(21, 26, "PRODUCT")]}),
    ("How much does a jacket cost?", {"entities": [(16, 22, "PRODUCT")]}),
    ("Tell me the cost of a hat", {"entities": [(20, 24, "PRODUCT")]}),
    ("What is the price of jeans?", {"entities": [(21, 26, "PRODUCT")]}),
    ("How much is a t-shirt?", {"entities": [(12, 19, "PRODUCT")]}),

    ("What is the price of a chair?", {"entities": [(23, 28, "PRODUCT")]}),
    ("How much does a table cost?", {"entities": [(16, 21, "PRODUCT")]}),
    ("Tell me the price of a sofa", {"entities": [(20, 25, "PRODUCT")]}),
    ("What is the cost of a bed?", {"entities": [(20, 23, "PRODUCT")]}),
    ("How much is a mattress?", {"entities": [(12, 20, "PRODUCT")]}),

    ("What is the price of soap?", {"entities": [(21, 25, "PRODUCT")]}),
    ("How much does toothpaste cost?", {"entities": [(14, 23, "PRODUCT")]}),
    ("Tell me the cost of shampoo", {"entities": [(20, 27, "PRODUCT")]}),
    ("What is the price of detergent?", {"entities": [(21, 31, "PRODUCT")]}),
    ("How much is handwash?", {"entities": [(12, 20, "PRODUCT")]}),

    ("What is the price of a TV?", {"entities": [(23, 25, "PRODUCT")]}),
    ("How much does a fridge cost?", {"entities": [(16, 22, "PRODUCT")]}),
    ("Tell me the price of a microwave", {"entities": [(20, 31, "PRODUCT")]}),
    ("What is the cost of a blender?", {"entities": [(20, 27, "PRODUCT")]}),
    ("How much is a cooker?", {"entities": [(12, 18, "PRODUCT")]}),

    ("What is the price of a watch?", {"entities": [(23, 28, "PRODUCT")]}),
    ("How much does a bag cost?", {"entities": [(16, 19, "PRODUCT")]}),
    ("Tell me the cost of sunglasses", {"entities": [(20, 30, "PRODUCT")]}),
    ("What is the price of a belt?", {"entities": [(23, 27, "PRODUCT")]}),
    ("How much is a wallet?", {"entities": [(12, 18, "PRODUCT")]}),
]

In [27]:
# load the spacy 
nlp = spacy.load("en_core_web_sm")

In [28]:
if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

In [29]:
for _,annotations in train_data:
    for ent in annotations["entities"]:
        if ent[2] not in ner.labels:
            ner.add_label(ent[2])

In [30]:
# disables other pipes 
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

In [34]:
# spaCy freezes all components except ner
with nlp.disable_pipes(*other_pipes):
    # spaCy initializes an optimizer
    optimizer = nlp.resume_training()
    # Sets learning rates
    # Prepares gradient tracking

    epochs = 50
    # One epoch = The model sees all training examples once
    # 50 epochs = Same data shown 50 times, shuffled each time
   

    # an epoch is a interation on the dataset 
    for epoch in range(epochs):
        # ===========================================================
        random.shuffle(train_data)
        # shuffling the data 
        # Neural networks are order-sensitive. If you don’t shuffle:
        # 1. Model memorizes patterns
        # 2. Learns bad shortcuts
        # 3. Overfits faster

        losses = {}

        #========================================================
        batches = minibatch(train_data, size=2)
        # dividing the training data into smaller, manageable chunks for each iteration of the training loop
        # model updates its weights after processing each mini-batch, rather than the entire dataset or a single example

        # =========================================================
        
        for batch in batches:
            examples = []
            for text,annotations in batch:
                # Converting text into Docs
                doc = nlp.make_doc(text)
                # Create Example (predicted vs gold)
                example = Example.from_dict(doc,annotations)
                examples.append(example)
            
             # Update the NER model
            nlp.update(examples,drop=0.5,losses=losses,sgd=optimizer)
            
        print(f"Epoch {epoch + 1} | Losses: {losses}")




c:\Users\nyaka\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "How much does toothpaste cost?" with entities "[(14, 23, 'PRODUCT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\nyaka\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Tell me the price of headphones" with entities "[(20, 30, 'PRODUCT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\nyaka\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in

Epoch 1 | Losses: {'ner': np.float32(49.129517)}
Epoch 2 | Losses: {'ner': np.float32(53.129932)}
Epoch 3 | Losses: {'ner': np.float32(32.041744)}
Epoch 4 | Losses: {'ner': np.float32(15.641085)}
Epoch 5 | Losses: {'ner': np.float32(16.451654)}
Epoch 6 | Losses: {'ner': np.float32(14.188722)}
Epoch 7 | Losses: {'ner': np.float32(4.588805)}
Epoch 8 | Losses: {'ner': np.float32(12.680008)}
Epoch 9 | Losses: {'ner': np.float32(8.519009)}
Epoch 10 | Losses: {'ner': np.float32(6.303315)}
Epoch 11 | Losses: {'ner': np.float32(11.997067)}
Epoch 12 | Losses: {'ner': np.float32(2.645238)}
Epoch 13 | Losses: {'ner': np.float32(6.2780304)}
Epoch 14 | Losses: {'ner': np.float32(4.309519)}
Epoch 15 | Losses: {'ner': np.float32(4.5808754)}
Epoch 16 | Losses: {'ner': np.float32(8.54203)}
Epoch 17 | Losses: {'ner': np.float32(4.1579156)}
Epoch 18 | Losses: {'ner': np.float32(1.9383609)}
Epoch 19 | Losses: {'ner': np.float32(4.553558)}
Epoch 20 | Losses: {'ner': np.float32(1.2246615)}
Epoch 21 | Losses

In [35]:
# save the model
nlp.to_disk("custom_ner_model")

# load the model
trained_ner_model = spacy.load("custom_ner_model")

In [ ]:
test_texts = [
    "I want to buy a chair ",
    "How much is the price of a Curtain",
    "Tell me the cost of a book",
    "How much a laptop ",
    "How much is a car",
    "What is the cost of a spoon"
]

for text in test_texts:
    doc = trained_ner_model(text)

    for ent in doc.ents:
        print(f"{ent.text} => {ent.label_}")
          

chair => PRODUCT
Curtain => PRODUCT
book => PRODUCT
laptop => PRODUCT
car => PRODUCT
spoon => PRODUCT
